In [1]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
import geopandas as gpd
import matplotlib.pyplot as plt
import folium
import plotly.express as px
from fuzzywuzzy import process
from shapely.geometry import Point
from shapely import wkt

c:\Users\User\AppData\Local\Programs\Python\Python311\Lib\site-packages\fuzzywuzzy\fuzz.py:11: UserWarning: Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning
  warnings.warn('Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning')


In [2]:
df_him = pd.read_csv(r'c:\Users\User\Documents\NIR3\files\Сообщения_для_сценария_2_geocoded.csv', sep=';')
df_him['geometry'] = df_him['geometry'].apply(wkt.loads)
gdf_points_him = gpd.GeoDataFrame(df_him, geometry='geometry', crs="EPSG:4326")

In [3]:
gdf = gpd.read_file(r'c:\Users\User\Documents\NIR3\files\MO.geojson', encoding='cp1251')
gdf["NAME"] = gdf["NAME"].apply(lambda val: val.replace("округ", "").strip())

In [4]:
gdf_joined = gpd.sjoin(
    gdf_points_him,
    gdf[['geometry', 'NAME']],
    how='left',
    predicate='within'
)
gdf_joined.rename(columns={'NAME':'loc_mo'})
gdf_joined

,Unnamed: 0,text,geometry,index_right,NAME
0,847,"На Мельникова, Пролетарской, Маяковского и в м...",POINT (30.35550 59.93682),77,Литейный
1,858,"Дискотека в башнях №33,35,37 на Мельникова 🪩\r...",POINT (30.31731 59.96071),68,Посадский
2,855,А вот так по окнам жилых домов бьет лазерная р...,POINT (30.09959 59.82870),19,Константиновское
3,884,От подписчицы:\r\n______________\r\nОгороженны...,POINT (30.11692 59.78970),18,Горелово
4,894,От подписчицы:\r\n_____________\r\nУл. Дружбы ...,POINT (30.07098 59.72384),17,Красное Село
5,929,В Юбилейном сквере на ул.Калинина уже подснежн...,POINT (30.25229 59.89620),47,Нарвский
6,950,От подписчиков:\r\n_____________\r\nПо адресу ...,POINT (30.35563 59.94029),77,Литейный
7,970,Картина маслом: «Уют по-химкински»\r\n\r\nПанф...,POINT (30.41482 59.95210),92,Большая Охта
8,992,Сквер на Энгельса 26 сегодня начали ограждать ...,POINT (30.33391 60.05122),74,Парнас
9,1094,Штраф до 5.000 рублей ждёт россиян за неправил...,POINT (30.21677 60.02436),37,Юнтолово


In [5]:
name_counts = gdf_joined['NAME'].value_counts().reset_index()
name_counts.columns = ['NAME', 'count']
name_counts

,NAME,count
0,Литейный,3
1,Посадский,1
2,Константиновское,1
3,Горелово,1
4,Красное Село,1
5,Нарвский,1
6,Большая Охта,1
7,Парнас,1
8,Юнтолово,1
9,Парголово,1


In [7]:
gdf_mer = gdf.merge(name_counts, left_on="NAME", right_on='NAME')

In [9]:
gdf_mer

,OSM_ID,NAME,ADMIN_LVL,Готово,geometry,count
0,-363092.0,Красное Село,8,None,"MULTIPOLYGON (((30.03496 59.71785, 30.03842 59...",1
1,-363095.0,Горелово,8,None,"MULTIPOLYGON (((30.04334 59.76418, 30.04535 59...",1
2,-363613.0,Константиновское,8,None,"MULTIPOLYGON (((30.06974 59.83842, 30.07552 59...",1
3,-1181031.0,Парголово,8,None,"MULTIPOLYGON (((30.13125 60.05112, 30.13451 60...",1
4,-1205337.0,Юнтолово,8,None,"MULTIPOLYGON (((30.18355 60.03227, 30.18363 60...",1
5,-1185367.0,Нарвский,8,None,"MULTIPOLYGON (((30.22333 59.88356, 30.22598 59...",1
6,-1187391.0,Посадский,8,None,"MULTIPOLYGON (((30.31451 59.96278, 30.31457 59...",1
7,-1181033.0,Парнас,8,None,"MULTIPOLYGON (((30.32823 60.04384, 30.33455 60...",1
8,-1198058.0,Литейный,8,None,"MULTIPOLYGON (((30.34600 59.94952, 30.34674 59...",3
9,-1185383.0,Большая Охта,8,None,"MULTIPOLYGON (((30.40344 59.95692, 30.40321 59...",1


In [ ]:
fig = px.choropleth_mapbox(gdf_mer, 
                           geojson=gdf_mer.geometry, 
                           locations=gdf_mer.index,
                           hover_name='NAME',
                           color='count', 
                           mapbox_style="carto-positron",
                           center={"lat": 59.9343, "lon": 30.3351},
                           zoom=8)
fig.update_layout(width=1200, height=800)
fig.show()